# 🏠 HDB Resale Data Engineering Pipeline
### End-to-End ETL, Data Quality, and Transformation Framework

**Author:** Parag  
**Role Applied:** Senior Data Engineer  
**Objective:** Build a robust, scalable, and explainable ETL pipeline for HDB resale data.

## 📌 Problem Overview

The objective is to design and implement a data engineering pipeline to:

- Ingest raw HDB resale datasets (2012–2016)
- Perform data profiling and validation
- Apply business rules and transformations
- Generate a unique resale identifier
- Detect anomalies and ensure data quality
- Output datasets into structured layers

---

## 🧠 Engineering Philosophy

This pipeline is designed with:

- **Modularity** → Each stage is independently testable  
- **Scalability** → Suitable for future cloud deployment  
- **Data Quality First** → Validation before transformation  
- **Reproducibility** → Fully programmatic, no manual intervention  

## Setup

In [2]:
import pandas as pd
import numpy as np
import requests
import hashlib
import re
import time
from datetime import datetime
from pathlib import Path

pd.set_option('display.max_columns', None)

## Data Extraction

In [3]:
# import time

BASE_DIR = Path("../").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
s = requests.Session()

DATASETS = [
    {
        "name": "ResaleFlatPricesBasedonApprovalDate19901999",
        "dataset_id": "d_ebc5ab87086db484f88045b47411ebc5",
    },
    {
        "name": "ResaleFlatPricesBasedonApprovalDate2000Feb2012",
        "dataset_id": "d_43f493c6c50d54243cc1eab0df142d6a",
    },
    {
        "name": "ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014",
        "dataset_id": "d_2d5ff9ea31397b66239f245f57751537",
    },
    {
        "name": "ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016",
        "dataset_id": "d_ea9ed51da2787afaf8e51f827c304208",
    },
    {
        "name": "ResaleflatpricesbasedonregistrationdatefromJan2017onwards",
        "dataset_id": "d_8b84c4ee58e3cfc0ece0d773c8ca6abc",
    },
]


def download_file(dataset_id, output_file):
    headers = {"Content-Type": "application/json"}

    init = s.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download",
        headers=headers,
        json={}
    )
    if init.status_code == 429:
        raise RuntimeError("Rate-limited on initiate-download; wait and retry")
    init.raise_for_status()

    max_polls = 8
    for i in range(max_polls):
        poll = s.get(
            f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download",
            headers=headers,
            json={}
        )
        if poll.status_code == 429:
            time.sleep(15)
            continue
        poll.raise_for_status()

        payload = poll.json()
        data = payload.get("data") or {}
        status = data.get("status")
        url = data.get("url")

        if status == "DOWNLOAD_SUCCESS" and url:
            df = pd.read_csv(url)
            print(df.head())   # instead of display(...)
            df.to_csv(output_file, index=False)
            print("Dataframe loaded and saved.")
            return

        print(f"{i+1}/{max_polls}: not ready yet ({status}), polling again...")
        time.sleep(4)

    raise RuntimeError(f"Download URL not ready for {dataset_id}")


def download_data():
    downloaded_files = []

    for dataset in DATASETS:
        # csv_url = get_csv_url(dataset["dataset_id"])
        DATASET_ID = dataset["dataset_id"]
        output_file = RAW_DIR / f"{dataset['name']}.csv"
            # download_csv(csv_url, output_file)
        #     download_file(DATASET_ID, output_file)
        #     downloaded_files.append(output_file)
        # except Exception as e:
        #     print(f"Skipping {dataset['name']}: {e}")
        max_attempts = 5
        for attempt in range(max_attempts):
            try:
                download_file(dataset["dataset_id"], output_file)
                downloaded_files.append(output_file)
                break
            except RuntimeError as e:
                if "Rate-limited" in str(e):
                    wait = 15 * (attempt + 1)  # 15s, 30s, 45s...
                    print(f"{dataset['name']}: rate-limited, waiting {wait}s before retry...")
                    time.sleep(wait)
                    continue
                raise

    print("Downloaded files:")
    for file in downloaded_files:
        print(file)

    for file in downloaded_files:
        df = pd.read_csv(file)
        print(f"{file.name}: shape={df.shape}, columns={list(df.columns)}")


download_data()

ResaleFlatPricesBasedonApprovalDate19901999: rate-limited, waiting 15s before retry...
     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06   

   floor_area_sqm      flat_model  lease_commence_date  resale_price  
0            31.0        IMPROVED                 1977          9000  
1            31.0        IMPROVED                 1977          6000  
2            31.0        IMPROVED                 1977          8000  
3            31.0        IMPROVED                 1977          6000  
4            73.0  NEW GENERATION                 1976         47200  
Dataframe loaded and saved.
ResaleFlatPricesBase

## Data Profiling

In [14]:
import glob
def load_and_combine(path=None):
    if path is None:
        # raw_dir =  "data/raw"
        path = "../data/raw/*.csv"
    files = glob.glob(path)
    if not files:
        raise FileNotFoundError(f"No CSV files found for pattern: {path}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    # print(df.head())
    # print(df.shape)
    # print(df.columns)
    # print(df.dtypes)
    # print(df.isnull().sum())
    # print(df.nunique())
    return df

def profile_data(df):
    profile = {
        "row_count": len(df),
        "columns": df.columns.tolist(),
        "missing": df.isnull().sum().to_dict(),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "unique_counts": df.nunique().to_dict()
    }
    return profile

df_raw = load_and_combine()
print(f"profile -> {profile_data(df_raw)}")

profile -> {'row_count': 974656, 'columns': ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease'], 'missing': {'month': 0, 'town': 0, 'flat_type': 0, 'block': 0, 'street_name': 0, 'storey_range': 0, 'floor_area_sqm': 0, 'flat_model': 0, 'lease_commence_date': 0, 'resale_price': 0, 'remaining_lease': 709050}, 'dtypes': {'month': 'str', 'town': 'str', 'flat_type': 'str', 'block': 'str', 'street_name': 'str', 'storey_range': 'str', 'floor_area_sqm': 'float64', 'flat_model': 'str', 'lease_commence_date': 'int64', 'resale_price': 'float64', 'remaining_lease': 'object'}, 'unique_counts': {'month': 436, 'town': 27, 'flat_type': 8, 'block': 2757, 'street_name': 595, 'storey_range': 25, 'floor_area_sqm': 228, 'flat_model': 34, 'lease_commence_date': 57, 'resale_price': 10209, 'remaining_lease': 748}}


## 🧪 Data Quality Framework with Great Expectations

Integrated Great Expectations to formalize and automate data validation.

Benefits:
- Declarative validation rules
- Reusable expectations
- Data quality reporting
- Pipeline integration ready

In [56]:
import great_expectations as gx

context = gx.get_context()

### Create Data Source

In [57]:
datasource = context.data_sources.add_pandas("hdb_datasource")
data_asset = datasource.add_dataframe_asset(name="hdb_data_asset")

### Create Expectation Suite

In [58]:
from great_expectations import ExpectationSuite

suite = context.suites.add_or_update(ExpectationSuite(name="hdb_suite"))

### Validator

In [59]:
batch_request = data_asset.build_batch_request(options={"dataframe": df_raw})

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=suite
)

### Add Expectations

#### Now we translate your validation logic into formal expectations:

### ✅ Date Format

In [60]:
validator.expect_column_values_to_match_regex(
    column="month",
    regex=r"\d{4}-\d{2}"
)

c:\Parag\Parag\IVs\hdb\hdb-etl-pipeline\.venv\Lib\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 94.57it/s]  


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_match_regex",
    "kwargs": {
      "batch_id": "hdb_datasource-hdb_data_asset",
      "column": "month",
      "regex": "\\d{4}-\\d{2}"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 974656,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### ✅ Town Not Null

In [61]:
validator.expect_column_values_to_not_be_null("town")

c:\Parag\Parag\IVs\hdb\hdb-etl-pipeline\.venv\Lib\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 6/6 [00:00<00:00, 545.18it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "batch_id": "hdb_datasource-hdb_data_asset",
      "column": "town"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 974656,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### ✅ Flat Type Valid Set

In [62]:
validator.expect_column_values_to_be_in_set(
    "flat_type",
    list(df_raw["flat_type"].dropna().unique())
)

c:\Parag\Parag\IVs\hdb\hdb-etl-pipeline\.venv\Lib\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics:  62%|██████▎   | 5/8 [00:00<00:00, 178.99it/s] 

Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 207.80it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "hdb_datasource-hdb_data_asset",
      "column": "flat_type",
      "value_set": [
        "1 ROOM",
        "3 ROOM",
        "4 ROOM",
        "5 ROOM",
        "2 ROOM",
        "EXECUTIVE",
        "MULTI GENERATION",
        "MULTI-GENERATION"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 974656,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### ✅ Storey Range Pattern

In [63]:
validator.expect_column_values_to_match_regex(
    "storey_range",
    r"\d+"
)

c:\Parag\Parag\IVs\hdb\hdb-etl-pipeline\.venv\Lib\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 104.09it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_match_regex",
    "kwargs": {
      "batch_id": "hdb_datasource-hdb_data_asset",
      "column": "storey_range",
      "regex": "\\d+"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 974656,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### ✅ Resale Price Range (Dynamic)

In [64]:
validator.expect_column_values_to_be_between(
    "resale_price",
    min_value=df_raw["resale_price"].quantile(0.01),
    max_value=df_raw["resale_price"].quantile(0.99)
)

c:\Parag\Parag\IVs\hdb\hdb-etl-pipeline\.venv\Lib\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 350.76it/s]

{
  "success": false,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "hdb_datasource-hdb_data_asset",
      "column": "resale_price",
      "min_value": 40000.0,
      "max_value": 910000.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 974656,
    "unexpected_count": 17695,
    "unexpected_percent": 1.8155123448683432,
    "partial_unexpected_list": [
      9000.0,
      6000.0,
      8000.0,
      6000.0,
      38000.0,
      35000.0,
      37000.0,
      37000.0,
      34000.0,
      36000.0,
      36000.0,
      37000.0,
      37000.0,
      34000.0,
      38000.0,
      37500.0,
      35000.0,
      35000.0,
      37000.0,
      37000.0
    ],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 1.8155123448683432,
    "unexpected_percent_nonmissing": 1.8155123448683432
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "

### 💾 Save Expectations

In [65]:
suite = validator.get_expectation_suite()
context.suites.add_or_update(suite)

{
  "name": "hdb_suite",
  "id": "66c7e9fe-4e28-4e63-b4c0-fae974e619d4",
  "expectations": [
    {
      "type": "expect_column_values_to_match_regex",
      "kwargs": {
        "column": "month",
        "regex": "\\d{4}-\\d{2}"
      },
      "meta": {},
      "id": "bae6f53b-731b-46c7-9e46-9b85c895c9da",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "town"
      },
      "meta": {},
      "id": "9fe0463c-285f-4c47-a062-c7da9871d139",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_in_set",
      "kwargs": {
        "column": "flat_type",
        "value_set": [
          "1 ROOM",
          "3 ROOM",
          "4 ROOM",
          "5 ROOM",
          "2 ROOM",
          "EXECUTIVE",
          "MULTI GENERATION",
          "MULTI-GENERATION"
        ]
      },
      "meta": {},
      "id": "adb2901e-67e7-4ba4-ab51-0baba2f13ab2",
      "severity": "critical"
    },
  

### 🚀 Run Validation

In [83]:
from great_expectations.checkpoint import Checkpoint
from great_expectations import ValidationDefinition

# 1. Get batch definition
# data_asset.add_batch_definition("hdb_batch")
batch_def = data_asset.get_batch_definition("hdb_batch")

# 2. Get suite
suite = context.suites.get("hdb_suite")

# 3. Create validation definition
validation_definition = ValidationDefinition(
    name="hdb_validation",
    data=batch_def,
    suite=suite
)

# 4. Register it
context.validation_definitions.add_or_update(validation_definition)

# 5. Create checkpoint
checkpoint = Checkpoint(
    name="hdb_checkpoint",
    validation_definitions=[validation_definition]
)

# 6. Register checkpoint
context.checkpoints.add_or_update(checkpoint)

# 7. Run checkpoint
results = checkpoint.run(
    batch_parameters={
        "dataframe": df_raw
    }
)

# 8. Build & open docs
context.build_data_docs()
context.open_data_docs()

Calculating Metrics: 100%|██████████| 39/39 [00:00<00:00, 119.05it/s]


## 📊 Data Quality Reporting (Great Expectations)

The pipeline integrates Great Expectations to provide automated, human-readable validation reports.

---

### 🔍 Validation Summary

The overview dashboard provides a quick snapshot of dataset health, including pass/fail status across all expectations.

[![Validation Overview](../screenshots/ge-overview.png)](gx/uncommitted/data_docs/local_site/index.html)

[![Validation Overview](../screenshots/ge-overview1.png)](gx/uncommitted/data_docs/local_site/index.html)

---

### ✅ Expectation-Level Results

Each validation rule is evaluated independently, enabling precise identification of data quality issues.

[![Expectation Results](../screenshots/ge-expectations.png)](gx/uncommitted/data_docs/local_site/index.html)

---

### 📈 Failures

.

[![Column Validation](../screenshots/ge-failures.png)](gx/uncommitted/data_docs/local_site/index.html)

---

## 🧪 Data Validation

Validation rules are derived dynamically from dataset characteristics:

### Rules:
- Date format: YYYY-MM
- Town: Non-null
- Flat Type: Valid categories
- Storey Range: Numeric pattern

In [15]:
def validate(df):
    failed = pd.DataFrame()
        # Example rules
    valid_df = df.copy()
    # Date format check
    valid_df = valid_df[valid_df['month'].str.match(r'\d{4}-\d{2}')]
    # Town non-null
    valid_df = valid_df[valid_df['town'].notnull()]
    # Flat type known values
    allowed_flat_types = df['flat_type'].dropna().unique()
    valid_df = valid_df[valid_df['flat_type'].isin(allowed_flat_types)]
    # Storey range format
    valid_df = valid_df[valid_df['storey_range'].str.contains(r'\d+', na=False)]
    # Split failed
    failed = df[~df.index.isin(valid_df.index)]

    return valid_df, failed

df_valid , df_failed_validation = validate(load_and_combine())

print("Valid:", len(df_valid), "Failed:", len(df_failed_validation))

Valid: 974656 Failed: 0


## 🏠 Remaining Lease Calculation

Assumption:
- All HDB leases are 99 years

Remaining lease is calculated dynamically based on current year.

In [16]:
def compute_remaining_lease(row):
    lease_start_year = int(row['lease_commence_date'])
    current_year = datetime.now().year
    
    remaining_years = 99 - (current_year - lease_start_year)
    
    years = max(remaining_years, 0)
    months = 0
    
    return f"{years} years {months} months"

df_valid['remaining_lease_calc'] = df_valid.apply(compute_remaining_lease, axis=1)
print(df_valid.head(5))

     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06   

   floor_area_sqm      flat_model  lease_commence_date  resale_price  \
0            31.0        IMPROVED                 1977        9000.0   
1            31.0        IMPROVED                 1977        6000.0   
2            31.0        IMPROVED                 1977        8000.0   
3            31.0        IMPROVED                 1977        6000.0   
4            73.0  NEW GENERATION                 1976       47200.0   

  remaining_lease remaining_lease_calc  
0             NaN    50 years 0 months  
1             NaN    50 years 0 months  
2    

## 🔁 Deduplication Strategy

Rule:
If all columns except resale_price are identical:
→ Keep the higher price  
→ Move lower price to failed dataset

In [17]:
def deduplicate(df):
    key_cols = [col for col in df.columns if col != 'resale_price']
    df_sorted = df.sort_values(by='resale_price', ascending=False)
    deduped = df_sorted.drop_duplicates(subset=key_cols, keep='first')
    failed = df[~df.index.isin(deduped.index)]
    return deduped, failed

df_dedup, df_dup_failed = deduplicate(df_valid)
print(f"df_dedup: {df_dedup.shape}")
print(f"df_dup_failed: {df_dup_failed.shape}")

print(f"df_dup_failed first 2 records: {df_dup_failed.head(2)}")

df_dedup: (945398, 12)
df_dup_failed: (29258, 12)
df_dup_failed first 2 records:       month        town flat_type block        street_name storey_range  \
2   1990-01  ANG MO KIO    1 ROOM   309   ANG MO KIO AVE 1     10 TO 12   
43  1990-01  ANG MO KIO    3 ROOM   442  ANG MO KIO AVE 10     07 TO 09   

    floor_area_sqm      flat_model  lease_commence_date  resale_price  \
2             31.0        IMPROVED                 1977        8000.0   
43            67.0  NEW GENERATION                 1979       40000.0   

   remaining_lease remaining_lease_calc  
2              NaN    50 years 0 months  
43             NaN    52 years 0 months  


## 🚨 Anomaly Detection

Method:
- IQR-based statistical approach

Why:
- Robust against skewed distributions
- Industry standard heuristic

In [10]:
q1 = df_dedup['resale_price'].quantile(0.25)
q3 = df_dedup['resale_price'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df_anomalies = df_dedup[
    (df_dedup['resale_price'] < lower) |
    (df_dedup['resale_price'] > upper)
]

print("Anomalies detected:", len(df_anomalies))

print("Anomalies detected - first two:", df_anomalies.head(2))


Anomalies detected: 23739
Anomalies detected - first two:           month        town flat_type block street_name storey_range  \
920016  2026-02  QUEENSTOWN    5 ROOM    92   DAWSON RD     19 TO 21   
906074  2025-06  QUEENSTOWN    5 ROOM    92   DAWSON RD     22 TO 24   

        floor_area_sqm              flat_model  lease_commence_date  \
920016           122.0  Premium Apartment Loft                 2016   
906074           122.0  Premium Apartment Loft                 2016   

        resale_price     remaining_lease remaining_lease_calc  
920016     1700000.0  89 years 03 months    89 years 0 months  
906074     1658888.0  89 years 11 months    89 years 0 months  


## 🔢 Resale Identifier Generation

Structure:

S + BlockDigits(3) + AvgPricePrefix(2) + Month(2) + TownInitial(1)

Example:
S1232301A

In [11]:
def generate_identifier(df):
    import re

    # Block digits
    df['block_digits'] = df['block'].apply(lambda x: re.sub(r'\D', '', str(x)).zfill(3)[:3])

    # Avg resale price grouping
    avg_price = df.groupby(['month', 'town', 'flat_type'])['resale_price'].mean().reset_index()
    avg_price['price_prefix'] = avg_price['resale_price'].astype(int).astype(str).str[:2]

    df = df.merge(avg_price[['month','town','flat_type','price_prefix']], on=['month','town','flat_type'])

    df['month_num'] = df['month'].str[-2:]
    df['town_char'] = df['town'].str[0]

    df['resale_id'] = (
        "S" +
        df['block_digits'] +
        df['price_prefix'] +
        df['month_num'] +
        df['town_char']
    )

    return df

df_transformed = generate_identifier(df_dedup)
print(f"df_transformed: {df_transformed.shape}")
print(f"df_transformed - first 2 records: {df_transformed.head(2)}")

df_transformed: (945398, 17)
df_transformed - first 2 records:      month        town flat_type block street_name storey_range  \
0  2026-02  QUEENSTOWN    5 ROOM    92   DAWSON RD     19 TO 21   
1  2025-06  QUEENSTOWN    5 ROOM    92   DAWSON RD     22 TO 24   

   floor_area_sqm              flat_model  lease_commence_date  resale_price  \
0           122.0  Premium Apartment Loft                 2016     1700000.0   
1           122.0  Premium Apartment Loft                 2016     1658888.0   

      remaining_lease remaining_lease_calc block_digits price_prefix  \
0  89 years 03 months    89 years 0 months          092           11   
1  89 years 11 months    89 years 0 months          092           12   

  month_num town_char  resale_id  
0        02         Q  S0921102Q  
1        06         Q  S0921206Q  


## 🔐 Hashing Strategy

Algorithm: SHA256

Why:
- Irreversible
- Collision-resistant
- Widely adopted standard

In [13]:
def hash_identifier(df):
    df['hashed_id'] = df['resale_id'].apply(
        lambda x: hashlib.sha256(x.encode()).hexdigest()
    )
    return df

df_hashed = hash_identifier(df_transformed)
print(f"df_hashed: {df_hashed.shape}")
print(f"first five df_hashed records : {df_hashed[['resale_id','hashed_id']].head(5)}")



df_hashed: (945398, 18)
first five df_hashed records :    resale_id                                          hashed_id
0  S0921102Q  fe8c51e42b8e500e229d8b880a01c0db4aff1fe768a8fe...
1  S0921206Q  b402eb5dc0d00f534459113590b8eb519ad7aba78a6d64...
2  S0091103B  864b855c982575e1fc600307c7fa7fc6f3c5ded1c000b8...
3  S2751111B  24ff15abe530361fcc5cef656eb1a34bf07a5b3517612a...
4  S1381101T  13c1fe55ac62d065f473d052bdc85ec5df64db787740d6...


## 💾 Output Data Layers

| Layer        | Description |
|-------------|------------|
| Raw         | Original dataset |
| Cleaned     | Validated + deduplicated |
| Transformed | Identifier added |
| Failed      | Invalid + duplicate records |
| Hashed      | Final dataset with hashed ID |

In [18]:
def save_outputs(df_raw, df_dedup, df_anomalies, df_transformed, df_hashed):
    for d in [df_raw, df_dedup, df_anomalies, df_transformed, df_hashed]:
        if 'remaining_lease' in d.columns:
            d['remaining_lease'] = d['remaining_lease'].astype('string')
    df_raw.to_parquet("../data/raw/raw.parquet")
    df_dedup.to_parquet("../data/cleaned/cleaned.parquet")
    df_anomalies.to_parquet("../data/failed/anomalies.parquet")
    df_transformed.to_parquet("../data/transformed/transformed.parquet")
    df_hashed.to_parquet("../data/hashed/hashed.parquet")
    print("Outputs saved successfully")

save_outputs(df_raw, df_dedup, df_anomalies, df_transformed, df_hashed)

Outputs saved successfully


## 🧠 Key Design Decisions

### 1. Why Pandas?
Efficient for medium-scale structured datasets and rapid prototyping.

### 2. Why IQR for anomalies?
Statistically robust and interpretable.

### 3. Why SHA256?
Ensures irreversible hashing and uniqueness preservation.

### 4. Why modular stages?
Allows future migration to Spark / Airflow / AWS Glue.

### 5. Why Parquet?
Columnar storage → better performance in analytics workloads.